# Quantum Fourier Transform (QFT): 3-Qubit Demonstration

## Objectives
- Implement a 3-qubit QFT and its inverse.
- Validate correctness by round-trip tests and phase-kickback on computational basis states.

## Setup
```python
import numpy as np
from math import pi
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity
import matplotlib.pyplot as plt
```


## Theory Snapshot
The QFT maps computational basis states $|x\rangle$ to Fourier basis states via controlled phase rotations and Hadamards. It is the core subroutine of algorithms such as phase estimation and order finding.

## Circuit / Model
We construct QFT(3) with controlled phase rotations, followed by qubit swaps to reverse bit order. The inverse QFT is its Hermitian adjoint.

In [ ]:
from qiskit import QuantumCircuit
from math import pi

def qft(n: int, do_swaps: bool = True) -> QuantumCircuit:
    qc = QuantumCircuit(n)

    # MSB -> LSB: apply H on target qubit j, then CPs from k<j to j with angle pi/2^(j-k)
    for j in range(n-1, -1, -1):        # j = n-1, ..., 0
        qc.h(j)
        for k in range(j-1, -1, -1):    # k = j-1, ..., 0
            qc.cp(pi / (2 ** (j - k)), k, j)

    if do_swaps:
        # Reverse qubit order so |q_{n-1} ... q_0> maps to natural integer k
        for i in range(n // 2):
            qc.swap(i, n - 1 - i)

    return qc
    

def iqft(n: int, do_swaps: bool = True) -> QuantumCircuit:
    return qft(n, do_swaps=do_swaps).inverse()

# Convenience 3-qubit wrappers
qft3  = lambda: qft(3, do_swaps=True)
iqft3 = lambda: iqft(3, do_swaps=True)

## Experiments
### Experiment A (Round-trip)
 Apply QFT followed by inverse QFT to random basis states and verify state recovery.

In [ ]:
from qiskit.quantum_info import Statevector, state_fidelity

# Experiment A: round-trip
def round_trip_ok(x: int) -> float:
    qc = QuantumCircuit(3)
    # prepare |x>
    # initialize from label (little-endian)
    label = format(x, '03b')  # reverse for Qiskit endianness
    sv = Statevector.from_label(label)
    sv = sv.evolve(qft3()).evolve(iqft3())
    # fidelity with original
    return float(state_fidelity(sv, Statevector.from_label(label)))

vals = [round_trip_ok(x) for x in range(8)]
vals

### Experiment B (Phase behavior)
Apply QFT to $|1\rangle$ and inspect the complex amplitudes. For a 3‑qubit QFT, the ideal output is
$\frac{1}{\sqrt{8}}\sum_{k=0}^{7} e^{2\pi i \cdot 1 \cdot k / 8}\,|k\rangle$, i.e., the 8th roots of unity.

In [ ]:
import numpy as np

# We'll prepare |001> (decimal 1) so that q0 is |1>

prep = QuantumCircuit(3)
prep.x(0)  # set q0 = |1>
sv_in = Statevector.from_instruction(prep)

sv_out  = sv_in.evolve(qft3())

amps = sv_out.data  # complex amplitudes for |000>..|111>
mags = np.abs(amps)  # magnitudes

# Remove global phase using complex normalization 
N = 8
k = np.arange(N)
x = 1
idx0 = int(np.argmax(np.abs(amps) > 1e-12))  # find first amplitude that’s not ~0
g = np.exp(-1j * np.angle(amps[idx0]))  # build a phase correction factor 
phi_obs = np.angle(amps * g)  # extract relative (global-phase-free) phases

expected = np.exp(+2j*np.pi * x * k / N) / np.sqrt(N)
phi_exp  = np.angle(expected * g)  # expected (global-phase-free) phases

# Wrapped RMS
wrap = lambda a: ((a + np.pi) % (2*np.pi)) - np.pi  # wrap differences to (−pi, pi]
rms  = float(np.sqrt(np.mean(wrap(phi_obs - phi_exp)**2)))
print("RMS (should be ~1e-12):", rms)

# Sanity: unwrapped slope should be ~ +pi/4
slope = (np.unwrap(phi_obs)[-1] - np.unwrap(phi_obs)[0]) / (N-1)
print("Mean phase step (rad):", slope)


### Metrics & Plots
Fidelity of round-trip per input basis state should be $\approx 1.0$ on an ideal simulator.

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(); plt.bar(range(8), vals); plt.xlabel('|x> (decimal)'); plt.ylabel('Fidelity after QFT+IQFT'); plt.title('QFT Round-Trip Fidelities (3 qubits)'); plt.grid(True, axis='y')
except Exception as e:
    print('Plot skipped:', e)

#### Plot magnitudes and phases

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6,5), constrained_layout=True)
    ks = np.arange(8)
    ax1.bar(ks, mags); ax1.set_title('QFT(|1>) magnitudes'); ax1.set_xlabel('Basis index k'); ax1.set_ylabel('|amp|')
    ax1.grid(True, axis='y')
    ax2.scatter(ks, phi_obs, label='observed', marker='o')
    ax2.scatter(ks, phi_exp, label='expected', marker='x')
    ax2.set_title('Phases (radians)'); ax2.set_xlabel('k'); ax2.set_ylabel('phase')
    ax2.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax2.set_yticklabels([r'$-\pi$', r'$-\pi/2$', '0', r'$\pi/2$', r'$\pi$'])
    ax2.legend()
except Exception as e:
    print("Plot skipped:", e)

## Results & Discussion
- Round-trip fidelities $\approx 1.0$ confirm a correct QFT/Inverse implementation.
- The magnitudes are approximately uniform (≈$1/\sqrt{8}$) and the phases progress by $2\pi/8$ per basis index $k$, matching the analytic form.
- On hardware or with noise models, phase errors degrade fidelity; depth-optimized QFT or approximation can mitigate cost.

## References
- Coppersmith, *An Approximate Fourier Transform Useful in Quantum Factoring* (1994)
- Nielsen & Chuang, Ch. 5
- IBM Qiskit Textbook — QFT